In [1]:
import pandas as pd
from amphibian import get_data_accessor

# Pandas display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 50)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
SIGNUP_URL_PATTERN = 'https://www.stitchfix.com/signup%'
START_DATE         = '2026-01-01'

In [2]:
query("""--sql
      DESCRIBE curated.product_tracking_events
"""
)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Column,Type,Extra,Comment
0,payload,varchar,,
1,active_session_id,varchar,,
2,type,varchar,,
3,schema,varchar,,
4,name,varchar,,
5,action_name,varchar,,
6,screen_view_schema,varchar,,
7,screen_view_name,varchar,,
8,screen_view_type,varchar,,
9,screen_view_pte_id,varchar,,


In [3]:
signup_page = f"""--sql
SELECT DISTINCT visitor_id, platform, url
FROM curated.product_tracking_events
WHERE date_in_utc >= DATE '{START_DATE}'
    --AND url LIKE '{SIGNUP_URL_PATTERN}'
LIMIT 100000
"""

signup_df = query(signup_page)
signup_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,visitor_id,platform,url
0,0041ff18-c1c5-49ee-8d87-93530dda6910,web,https://www.stitchfix.com/d/gateway?msclkid=2d...
1,0041ff18-c1c5-49ee-8d87-93530dda6910,web,https://www.stitchfix.com/orders
2,0041ff18-c1c5-49ee-8d87-93530dda6910,web,https://www.stitchfix.com/tracking?carrier=ups...
3,00705249-ef33-42e5-b2f0-d80720ba1c7d,web,https://www.stitchfix.com/women?utm_source=boo...
4,0089e6d2-dad6-4da5-8c68-6e2341778196,iOS,None
...,...,...,...
99995,0c15978c-5bbe-42d7-a075-a51f3a8f73aa,web,https://www.stitchfix.com/quiz/shorts-style
99996,0c15978c-5bbe-42d7-a075-a51f3a8f73aa,web,https://www.stitchfix.com/quiz/shirts-style
99997,0c15978c-5bbe-42d7-a075-a51f3a8f73aa,web,https://www.stitchfix.com/quiz/fit-recap
99998,0c15978c-5bbe-42d7-a075-a51f3a8f73aa,web,https://www.stitchfix.com/quiz/style-intro


In [4]:
signup_df['url'].filter(signup_df['platform'] == 'iOS').unique()

array([], dtype=object)

In [5]:
signup_df.loc[signup_df['url'].str.contains('https://www.stitchfix.com/signup', na=False), 'url'].unique()

array(['https://www.stitchfix.com/signup',
       'https://www.stitchfix.com/signup?utm_campaign=social%7Cfacebook%7Cwomens%7Cw%7Cfix%7Cpros%7Cweb%7Cus%7Cconsideration%7Cstyleprofilecomplete%7Csepfy26&utm_medium=Instagram_Stories%7C090525%7C25-65%7Cw%7Cfb_ig%7Cadv%7Chighest-volume%7Cpros-broad%7Cvalue-pixel%7Ccomplete-registration%7Cstatic&utm_content=102925%7CStatic%7CStandard%7Cstylefilerusticrebel%7CStitchFix%7CNA%7CAlwaysOn%7CFY26%7CMix%7COFILD%7CDiscoveryourstylepersonalitywithStyleFileGetinspiredwithcuratedpicksfromaStylistbasedonyouruniquestyle%7CPersonalstylemadeeasy%7CNosubscriptionrequired%7CLearnMore%7Cwomens%7Chome%7C102925%7CNA%7C9x16-4x5&utm_term=120238555149740667&utm_source=ig&utm_id=120235348689640667&fbclid=PAdGRleAPMQJpleHRuA2FlbQEwAGFkaWQBqyxnYqD5-3NydGMGYXBwX2lkDzEyNDAyNDU3NDI4NzQxNAABp85Ja7cIxWkGOWR30hQkcu8k0s2QybVzGOMC_0Y_NkTUZC95vZQPhxxR-9y6_aem_U0_hyd8_5NVC6HONUPmA9g',
       'https://www.stitchfix.com/signup?ttclid=E_C_P_Cr8Bjj2l-nllBKQqA9dlNUseInzOMZCS94Ixj71

In [6]:
visitor_flags = f"""--sql
SELECT *
FROM curated.user_session_conversion_metrics cm
WHERE cm.region = 'US'
    AND cm.date_in_utc >= DATE '{START_DATE}'
LIMIT 5
"""

query(visitor_flags)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,client_id,visitor_id,active_session_id,inferred_client_flag,signed_in_session_flag,user_agent,screen_view_name,schema,name,type,url,referrer,utm_campaign,utm_medium,utm_source,utm_term,utm_content,lt_campaign,lt_medium,lt_source,lt_keyword,lt_content,lt_marketing_channel_id,region,landing_page,referrer_domain,test_flag,page_group_level_1,page_group_level_2,page_business_line,signup_ts,signup_past_flag,signup_in_session_flag,style_profile_completed_ts,style_profile_completed_past_flag,style_profile_completed_in_session_flag,style_shuffle_ts_in_session_flag,order_direct_buy_in_session_flag,checkout_fix_in_session_flag,order_fix_in_session_flag,datetime_in_utc,bounce_flag,session_length_in_sec,first_direct_order_flag,order_direct_buy_past_flag,first_fix_checkout_flag,signup_2d_flag,signup_7d_flag,style_profile_2d_flag,style_profile_7d_flag,request_2d_flag,request_7d_flag,order_direct_buy_2d_flag,order_direct_buy_7d_flag,order_direct_buy_20d_flag,checkout_fix_20d_flag,checkout_fix_30d_flag,checkout_fix_40d_flag,conversion_user_id,entry_door,utm_campaign_id,utm_adgroup_id,utm_ad_id,utm_keyword_id,utm_product_id,utm_site_id,lt_campaign_id,lt_adgroup_id,lt_ad_id,lt_keyword_id,lt_product_id,lt_site_id,date_in_utc
0,3001560,f9e49337-e287-4c55-8a1a-3c8bc020a65e,706b7547-e840-43cb-be27-9776d9091036,0,1,Mozilla/5.0 (Linux; Android 16; SM-S908U Build...,fix_home,screen_view,fix_home,screen_view,https://www.stitchfix.com/fix-home?sf_client_e...,None,email_us_n_transactional_fiatbs,email,blueshift,None,email_us_n_transactional_fiatbs_1209875442425662,email_us_n_transactional_fiatbs,email,blueshift,None,email_us_n_transactional_fiatbs_1209875442425662,001ea2c05a00341c7b613f1ddfa2b9a4,US,https://www.stitchfix.com/fix-home,None,0,fix-home,None,None,2011-11-13 08:00:00.000,1,0,2014-01-14 01:11:24.397,1,0,0,0,0,0,2026-04-27 01:19:00.417,0,156,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,3001560,front_door,None,None,None,None,None,None,None,None,None,None,None,None,2026-04-27
1,3017007,5359a44b-1878-4f59-8d2b-a645bd032ead,7EE010D0-50A2-4F2C-80A3-30E62449CB6F,0,1,StitchFix/10.79.0 (com.stitchfix.StitchFix-App...,category,category_screen_view,category,screen_view,None,None,push_us_w_freestyle_dresses_042726,push,blueshift,None,push_us_w_freestyle_dresses_042726,push_us_w_freestyle_dresses_042726,push,blueshift,None,push_us_w_freestyle_dresses_042726,f7dae1e829dd135c62ce656056aa519c,US,None,None,0,None,None,None,2012-06-07 07:00:00.000,1,0,2019-08-20 21:51:29.145,1,0,0,0,0,0,2026-04-27 17:09:01.467,0,16,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,3017007,front_door,None,None,None,None,None,None,None,None,None,None,None,None,2026-04-27
2,3033840,7c18a4db-6750-42fc-9a18-22d0eed7590a,11C69309-75D4-4F41-9C75-AFD2A2E129DC,0,1,StitchFix/11.1.0 (com.stitchfix.StitchFix-Apps...,homefeed,screen_view,homefeed,screen_view,None,None,None,None,None,None,None,None,None,None,None,None,54c24433c109fa0b2ad2f099de7b3758,US,None,None,0,None,None,None,2012-11-13 08:00:00.000,1,0,2015-02-22 15:37:28.341,1,0,0,0,0,0,2026-04-27 19:22:53.268,0,186,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,3033840,front_door,None,None,None,None,None,None,None,None,None,None,None,None,2026-04-27
3,3141428,5fa62bad-40b7-4d48-a823-cf359522957c,0A7A97B4-4368-42B1-9335-0DD8610F8B04,0,1,StitchFix/11.1.0 (com.stitchfix.StitchFix-Apps...,homefeed,screen_view,homefeed,screen_view,None,None,None,None,None,None,None,None,None,None,None,None,54c24433c109fa0b2ad2f099de7b3758,US,None,None,0,None,None,None,2013-07-27 01:59:42.298,1,0,2013-07-27 02:05:52.794,1,0,1,0,0,0,2026-04-27 02:37:39.894,0,385,0,1,0,0,0,0,0,0,0,0,0,0,1,1,1,3141428,front_door,None,None,None,None,None,None,None,None,None,None,None,None,2026-04-27
4,3141428,5fa62bad-40b7-4d48-a823-cf359522957c,982855EA-B47C-43CF-868A-126B47EBE324,0,1,StitchFix/11.1.0 (com.stitchfix.StitchFix-Apps...,homefeed,screen_view,homefeed,screen_view,None,None,None,None,None,None,None,None,None,None,None,None,54c24433c109fa0b2ad2f099de7b3758,US,None,None,0,None,None,None,2013-07-27 01:59:42.298,1,0,2013

In [7]:
month_days = f"""--sql
WITH visitor_flags AS (
    SELECT date_in_utc AS day
    FROM curated.user_session_conversion_metrics cm
    WHERE cm.region = 'US'
        AND cm.date_in_utc >= DATE '{START_DATE}'
)
SELECT DATE_TRUNC('month', day) AS month, COUNT(DISTINCT day) AS days_observed
FROM visitor_flags
GROUP BY 1
"""

query(month_days)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed
0,2026-04-01,30
1,2026-01-01,31
2,2026-06-01,30
3,2026-03-01,31
4,2026-02-01,28
5,2026-05-01,31


In [8]:
traffic_sql = f"""--sql
WITH signup_page AS (
    -- FLAW #1: eligibility accumulates from START_DATE (not tied to a specific visit month).
    SELECT DISTINCT visitor_id
    FROM curated.product_tracking_events
    WHERE date_in_utc >= DATE '{START_DATE}'
      AND url LIKE '{SIGNUP_URL_PATTERN}'
),
visitor_flags AS ( -- one row per visitor; new-visitor + signup status
    SELECT
        cm.visitor_id,
        MAX(cm.date_in_utc) AS day,   -- FLAW #3: month = visitor's LAST session, not their signup-page visit -> leakage
        -- FLAW #2: flags MAX()-ed across every in-window session, and `signup` has no conversion-window
        --          bound, so the values depend on which sessions fall in the window (i.e. on START_DATE).
        MAX(CASE WHEN cm.signup_ts IS NULL OR cm.signup_ts >= cm.datetime_in_utc THEN 1 ELSE 0 END) AS new_visitor,
        MAX(CASE WHEN cm.signup_ts >= cm.datetime_in_utc THEN 1 ELSE 0 END) AS signup
    FROM curated.user_session_conversion_metrics cm
    JOIN signup_page sp ON sp.visitor_id = cm.visitor_id
    WHERE cm.region = 'US'
      AND cm.date_in_utc >= DATE '{START_DATE}'
    GROUP BY cm.visitor_id
),
month_days AS (
    SELECT DATE_TRUNC('month', day) AS month, COUNT(DISTINCT day) AS days_observed
    FROM visitor_flags GROUP BY 1
)
SELECT
    DATE_TRUNC('month', vf.day) AS month,
    md.days_observed,
    SUM(new_visitor) AS signup_page_visitors,
    SUM(signup) AS signups,
    CAST(SUM(signup) AS DOUBLE) / NULLIF(SUM(new_visitor), 0) AS signup_page_conv_rate,
    ROUND(SUM(new_visitor) / md.days_observed, 1) AS signup_page_visitors_per_day
FROM visitor_flags vf
JOIN month_days md ON DATE_TRUNC('month', vf.day) = md.month
GROUP BY DATE_TRUNC('month', vf.day), md.days_observed
ORDER BY month DESC
"""

traffic_df = query(traffic_sql)
traffic_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,month,days_observed,signup_page_visitors,signups,signup_page_conv_rate,signup_page_visitors_per_day
0,2026-06-01,30,260456,142453,0.546937,8681
1,2026-05-01,31,279526,148259,0.530394,9016
2,2026-04-01,30,288482,154155,0.534366,9616
3,2026-03-01,31,356404,193023,0.541585,11496
4,2026-02-01,28,314835,172889,0.549142,11244
5,2026-01-01,31,317639,172658,0.543567,10246


**Original signup-page conversion query — kept here for inspection ONLY.**

NOT used in power_analysis.ipynb; superseded there by the visit-anchored version.

Flaws (all stem from working at the SESSION grain instead of the signup-page-VISIT grain):
1. Eligibility (`signup_page`) accumulates from START_DATE — a visitor counts toward every month in the window once they hit the signup page at any point, not just the month they did.
2. new_visitor / signup are MAX()-aggregated over ALL of a visitor's in-window sessions, and `signup` has no conversion-window bound — so widening the window can flip these flags.
3. Visitors are bucketed by MAX(date_in_utc) (their LAST session), not by their actual signup-page visit — so a given month's value drifts as START_DATE changes (cross-month leakage).
    => The monthly series is not self-contained or comparable; only the most recent fully-matured
       month is roughly trustworthy.

**Why it ultimately doesn't matter much:** the leakage shifts the baseline by only ~1-1.5pp
(e.g. 0.516 here vs 0.528 visit-anchored), which moves the required per-arm n by ~6%
(~half a day of duration). Fine for a quick sanity check — but not reproducible/defensible
enough for the documented power analysis, which is why the visit-anchored query is used instead.

================================

# Experiment Review

In [9]:
# Active Experiments

query(f"""--sql
DESCRIBE metrics.active_expt_settings
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Column,Type,Extra,Comment
0,expt_id,integer,,
1,plan_ids,array(varchar),,
2,start_date,date,,
3,end_date,date,,
4,start_fix_timestamp_field,varchar,,
5,end_fix_timestamp_field,varchar,,
6,metric_keys,array(varchar),,
7,as_of,varchar,partition key,


In [11]:
query(f"""--sql
SELECT COUNT(DISTINCT expt_id) FROM metrics.active_expt_settings
""")

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,_col0
0,356


In [15]:
query(
    f"""--sql
    SELECT *
    FROM metrics.active_expt_settings
    LIMIT 1
"""
)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,expt_id,plan_ids,start_date,end_date,start_fix_timestamp_field,end_fix_timestamp_field,metric_keys,as_of
0,882,[53e68372-df02-4389-84ed-d2b40b9671fe],2022-08-17,2023-02-23,first_preview_item_created_ts,styled_ts,"[client_combined_net_revenue_upto90d, client_f...",20221208


In [16]:
metric_counts_sql = f"""--sql
SELECT k AS metric_key, COUNT(DISTINCT expt_id) AS n_experiments
FROM metrics.active_expt_settings
CROSS JOIN UNNEST(metric_keys) AS t(k)
GROUP BY k
ORDER BY n_experiments DESC, metric_key
"""
metric_counts_df = query(metric_counts_sql)
metric_counts_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,metric_key,n_experiments
0,shipment_client_grp_fix_keep_rate_v2,147
1,client_fix_autoship_flag,141
2,client_fix_autoship_churn_flag,123
3,shipment_client_grp_fix_successful_fix_flag,109
4,shipment_client_grp_fix_order_value_v2,93
5,shipment_client_grp_fix_buy_zero_flag,75
6,client_linked_combined_net_revenue_upto30d,66
7,client_fix_cancellations,55
8,client_successful_first_fix_flag,55
9,client_fix_net_revenue,54
